In [1]:
import numpy as np
import pandas as pd
import os
import scipy.stats as stats
import scipy.signal as sig
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.multivariate.manova import MANOVA


from bokeh.io import push_notebook, show, output_notebook
from bokeh.layouts import row
from bokeh.plotting import figure
from bokeh.palettes import Plasma256, viridis, Category20
from bokeh.models import ColumnDataSource, SingleIntervalTicker, BasicTickFormatter
from bokeh.layouts import gridplot
from bokeh.models import Legend, LegendItem


output_notebook()

Loading BokehJS ...

In [2]:
df = pd.read_csv("TOF_raw_data_df.csv", 
                 parse_dates=['Date'],
    date_format="%y-%m-%d-%H-%M-%S")
print(df.shape)
print(df.head())

(13371, 140)
                                             Hash_id                Date  \
0  2d3a584ba1849ed6ea99636f22c8ace95476a3f9cea613... 2025-05-09 20:26:30   
1  2d3a584ba1849ed6ea99636f22c8ace95476a3f9cea613... 2025-05-09 20:26:30   
2  2d3a584ba1849ed6ea99636f22c8ace95476a3f9cea613... 2025-05-09 20:26:30   
3  2d3a584ba1849ed6ea99636f22c8ace95476a3f9cea613... 2025-05-09 20:26:30   
4  2d3a584ba1849ed6ea99636f22c8ace95476a3f9cea613... 2025-05-09 20:26:30   

           Test Labware_Name         Stacker_SN Axis Platform_Position  \
0  TOF BASELINE     baseline  FSTA1020250401005    z           retract   
1  TOF BASELINE     baseline  FSTA1020250401005    z           retract   
2  TOF BASELINE     baseline  FSTA1020250401005    z           retract   
3  TOF BASELINE     baseline  FSTA1020250401005    z           retract   
4  TOF BASELINE     baseline  FSTA1020250401005    z           retract   

   Labware_Num_X  Labware_Num_Z  Sample  ...    119    120    121    122  \
0        

In [11]:
p1 = figure(width=1200,height=800, title="Z Axis Baseline with 6 STD envelope, Platform extended with no labware")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]

# Plot the raw data
color_count = 0
stackers = df.query("Test!='TOF BASELINE'")['Stacker_SN'].unique().tolist()
for stacker in stackers:
    ys = df.query("Test!='TOF BASELINE' & Labware_Name=='baseline' & Axis=='z' & Platform_Position=='extend' & Labware_Num_X==0 & Zone==1 & Stacker_SN==@stacker")[bin_range].values.tolist()
    # ys = df.query("Test!='TOF BASELINE' & Labware_Name=='tiprack' & Axis=='z' & Platform_Position=='retract' & Labware_Num_X==1 & Labware_Num_Z==3 & Zone==1 & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    color = Plasma256[color_count]
    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color=color,
                    line_dash='solid',
                    legend_label=stacker)
    color_count += 10
    
# Create the baseline
df_query = "Test!='TOF BASELINE' & Labware_Name=='baseline' & Axis=='z' & Platform_Position=='extend' & Labware_Num_X==0 & Zone==1"
# df_query += " & Stacker_SN in ['FSTA1020250402003']"

baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
mean_value = baseline_data.mean()
baseline = baseline_data.mean()+(baseline_data.std()*5)
baseline_block = baseline_data.mean()-(baseline_data.std()*5)


p1.line(bin_range, mean_value, line_color='cyan', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline, line_color='green', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_block, line_color='red', line_width=2, line_dash='dashed')


p1.legend.click_policy="hide"
show(p1)

Platform Home Baseline Samples: 90


In [18]:
zone = 5
platform = 'retract'

zone = str(zone)
p1 = figure(width=1200,height=800, title=f"X Axis Baseline with 6 STD envelope, Platform {platform}ed, Zone {zone}")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]

# Plot the raw data
color_count = 0
stackers = df.query("Test!='TOF BASELINE'")['Stacker_SN'].unique().tolist()
for stacker in stackers:
    ys = df.query(f"Test!='TOF BASELINE' & Labware_Name=='nest-96-pcr' & Axis=='x' & Platform_Position=='{platform}' & Labware_Num_X==1 & Zone=={zone} & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    color = Plasma256[color_count]
    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color=color,
                    line_dash='solid',
                    legend_label=stacker)
    color_count += 10
    
# Create the baseline
df_query = f"Test!='TOF BASELINE' & Axis=='x' & Platform_Position=='{platform}' & Labware_Num_X==0 & Zone=={zone}"
baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
baseline = baseline_data.mean()+(baseline_data.std()*4)
baseline_block = baseline_data.mean()-(baseline_data.std()*4)


p1.line(bin_range, baseline, line_color='green', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_block, line_color='red', line_width=2, line_dash='dashed')

p1.legend.click_policy="hide"
show(p1)

Platform Home Baseline Samples: 90


In [19]:
zone = 6
platform = 'retract'

zone = str(zone)
p1 = figure(width=1200,height=800, title=f"X Axis Baseline with 6 STD envelope, Platform {platform}ed, Zone {zone}")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]

# Plot the raw data
color_count = 0
stackers = df.query("Test!='TOF BASELINE'")['Stacker_SN'].unique().tolist()
for stacker in stackers:
    ys = df.query(f"Test!='TOF BASELINE' & Labware_Name=='nest-96-pcr' & Axis=='x' & Platform_Position=='{platform}' & Labware_Num_X==1 & Zone=={zone} & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    color = Plasma256[color_count]
    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color=color,
                    line_dash='solid',
                    legend_label=stacker)
    color_count += 10
    
# Create the baseline
df_query = f"Test!='TOF BASELINE' & Axis=='x' & Platform_Position=='{platform}' & Labware_Num_X==0 & Zone=={zone}"
baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
baseline = baseline_data.mean()+(baseline_data.std()*4)
baseline_block = baseline_data.mean()-(baseline_data.std()*4)


p1.line(bin_range, baseline, line_color='green', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_block, line_color='red', line_width=2, line_dash='dashed')

p1.legend.click_policy="hide"
show(p1)

Platform Home Baseline Samples: 90


In [17]:
zone = 7
platform = 'retract'

zone = str(zone)
p1 = figure(width=1200,height=800, title=f"X Axis Baseline with 6 STD envelope, Platform {platform}ed, Zone {zone}")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]

# Plot the raw data
color_count = 0
stackers = df.query("Test!='TOF BASELINE'")['Stacker_SN'].unique().tolist()
for stacker in stackers:
    ys = df.query(f"Test!='TOF BASELINE' & Labware_Name=='nest-96-pcr' & Axis=='x' & Platform_Position=='{platform}' & Labware_Num_X==1 & Zone=={zone} & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    color = Plasma256[color_count]
    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color=color,
                    line_dash='solid',
                    legend_label=stacker)
    color_count += 10
    
# Create the baseline
df_query = f"Test!='TOF BASELINE' & Axis=='x' & Platform_Position=='{platform}' & Labware_Num_X==0 & Zone=={zone}"
baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
baseline = baseline_data.mean()+(baseline_data.std()*6)
baseline_block = baseline_data.mean()-(baseline_data.std()*6)


p1.line(bin_range, baseline, line_color='green', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_block, line_color='red', line_width=2, line_dash='dashed')

df_query = f"Test!='TOF BASELINE' & Axis=='x' & Platform_Position=='{platform}' & Labware_Num_X==1 & Zone=={zone}"
baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
lab_baseline = baseline_data.mean()-(baseline_data.std()*2)
p1.line(bin_range, lab_baseline, line_color='cyan', line_width=2, line_dash='dashed')

p1.legend.click_policy="hide"
show(p1)

Platform Home Baseline Samples: 90
Platform Home Baseline Samples: 120


In [31]:
zone = 4
platform = 'extend'

zone = str(zone)
p1 = figure(width=1200,height=800, title=f"X Axis Baseline with 6 STD envelope, Platform {platform}ed, Zone {zone}")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]

# Plot the raw data
color_count = 0
stackers = df.query("Test!='TOF BASELINE'")['Stacker_SN'].unique().tolist()
for stacker in stackers:
    ys = df.query(f"Test!='TOF BASELINE' & Labware_Name=='nest-96-pcr' & Axis=='x' & Platform_Position=='{platform}' & Labware_Num_X==1 & Zone=={zone} & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    color = Plasma256[color_count]
    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color=color,
                    line_dash='solid',
                    legend_label=stacker)
    color_count += 10
    
# Create the baseline
df_query = f"Test!='TOF BASELINE' & Axis=='x' & Platform_Position=='{platform}' & Labware_Num_X==0 & Zone=={zone}"
baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
baseline = baseline_data.mean()+(baseline_data.std()*3)
baseline_min = baseline_data.min()


p1.line(bin_range, baseline, line_color='green', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_min, line_color='red', line_width=2, line_dash='dashed')

df_query = f"Test!='TOF BASELINE' & Labware_Name=='nest-96-pcr' & Axis=='x' & Platform_Position=='{platform}' & Labware_Num_X==1 & Zone=={zone}"
baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
lab_baseline = baseline_data.mean()-(baseline_data.std()*2)
p1.line(bin_range, lab_baseline, line_color='cyan', line_width=2, line_dash='dashed')

p1.legend.click_policy="hide"
show(p1)

Platform Home Baseline Samples: 90
Platform Home Baseline Samples: 90


In [12]:
p1 = figure(width=1200,height=1000, title="Z Axis Labware with 4 STD envelope, Platform extended, 1 Nest 96 Well PCR Plate in Z")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]

# Plot the raw data
color_count = 0
stackers = df.query("Test!='TOF BASELINE'")['Stacker_SN'].unique().tolist()
for stacker in stackers:
    query_data = df.query("Test!='TOF BASELINE' & Labware_Name=='nest-96-pcr' & Axis=='z' & Platform_Position=='extend' & Labware_Num_Z==1 & Zone==1 & Stacker_SN==@stacker")
    ys = query_data[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines
    label_string = str(stacker) + ' Labware'
    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color='orange',
                    line_dash='solid',
                    legend_label=label_string)

    ys = df.query("Test!='TOF BASELINE' & Labware_Name=='baseline' & Axis=='z' & Platform_Position=='extend' & Labware_Num_Z==0 & Zone==1 & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    label_string = str(stacker) + ' Base'
    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color='blue',
                    line_dash='solid',
                    legend_label=label_string)

# Create the baseline
df_query = "Test!='TOF BASELINE' & Labware_Name=='baseline' & Axis=='z' & Platform_Position=='extend' & Labware_Num_X==0 & Zone==1"
# df_query += " & Stacker_SN in ['FSTA1020250409001','FSTA1020250408008','FSTA1020250408007','FSTA1020250403007','FSTA1020250407007','FSTA1020250403001','FSTA1020250407009','FSTA1020250403006','FSTA1020250402008','FSTA1020250402006','FSTA1020250402007','FSTA1020250401006','FSTA1020250403003','FSTA1020250407002','FSTA1020250407005','FSTA1020250407003','FSTA1020250403005','FSTA1020250402002']"

baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
baseline_mean = baseline_data.mean()
baseline = baseline_data.mean()+(baseline_data.std()*3)
baseline_block = baseline_data.mean()-(baseline_data.std()*3)
baseline_max = baseline_data.max()
baseline_median = baseline_data.median()+(baseline_data.std()*3)



p1.line(bin_range, baseline, line_color='green', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_block, line_color='red', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_max, line_color='purple', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_mean, line_color='cyan', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_median, line_color='magenta', line_width=2, line_dash='dashed')

p1.legend.click_policy="hide"
show(p1)

Platform Home Baseline Samples: 90


In [43]:
p1 = figure(width=1200,height=1000, title="Z Axis Labware with 4 STD envelope, Platform extended, 1 Nest 96 Well PCR Plate in Z")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]

# Plot the raw data
color_count = 0
stackers = df.query("Test!='TOF BASELINE'")['Stacker_SN'].unique().tolist()
for stacker in stackers:
    query_data = df.query("Test!='TOF BASELINE' & Labware_Name=='nest-96-pcr' & Axis=='z' & Platform_Position=='extend' & Labware_Num_Z==1 & Zone==1 & Stacker_SN==@stacker")
    colors = ['red', 'green', 'black']
    for test_date, tmp_c in zip(query_data['Date'].dt.date.unique(), colors):
        ys = query_data.query("Date.dt.date==@test_date")[bin_range].values.tolist()
        numlines = len(ys)
        xs = [bin_range]*numlines
        label_string = str(stacker) + ' ' + str(test_date) + ' Labware'
        p1.multi_line(xs = xs,
                        ys = ys,
                        line_color=tmp_c,
                        line_dash='solid',
                        legend_label=label_string)

        ys = df.query("Test!='TOF BASELINE' & Labware_Name=='baseline' & Axis=='z' & Platform_Position=='extend' & Labware_Num_Z==0 & Zone==1 & Stacker_SN==@stacker")[bin_range].values.tolist()
        numlines = len(ys)
        xs = [bin_range]*numlines

        label_string = str(stacker) + ' ' + str(test_date) + ' Base'
        p1.multi_line(xs = xs,
                        ys = ys,
                        line_color='blue',
                        line_dash='dashed',
                        legend_label=label_string)

# Create the baseline
df_query = "Test!='TOF BASELINE' & Labware_Name=='baseline' & Axis=='z' & Platform_Position=='extend' & Labware_Num_X==0 & Zone==1"
# df_query += " & Stacker_SN in ['FSTA1020250409001','FSTA1020250408008','FSTA1020250408007','FSTA1020250403007','FSTA1020250407007','FSTA1020250403001','FSTA1020250407009','FSTA1020250403006','FSTA1020250402008','FSTA1020250402006','FSTA1020250402007','FSTA1020250401006','FSTA1020250403003','FSTA1020250407002','FSTA1020250407005','FSTA1020250407003','FSTA1020250403005','FSTA1020250402002']"

baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
baseline_mean = baseline_data.mean()
baseline = baseline_data.mean()+(baseline_data.std()*4)
baseline_block = baseline_data.mean()-(baseline_data.std()*4)
baseline_max = baseline_data.max()
baseline_median = baseline_data.median()+(baseline_data.std()*4)



p1.line(bin_range, baseline, line_color='green', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_block, line_color='red', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_max, line_color='purple', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_mean, line_color='cyan', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_median, line_color='magenta', line_width=2, line_dash='dashed')

p1.legend.click_policy="hide"
show(p1)

Platform Home Baseline Samples: 90


Baseline with 

In [47]:
p1 = figure(width=1200,height=800, title="Z Axis Labware with 6 STD envelope, Platform extended with no labware")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]
    
# Create the baseline
df_query = "Test!='TOF BASELINE' & Labware_Name=='baseline' & Axis=='z' & Platform_Position=='extend' & Labware_Num_X==0 & Zone==1"
# df_query += " & Stacker_SN in ['FSTA1020250409001','FSTA1020250408008','FSTA1020250408007','FSTA1020250403007','FSTA1020250407007','FSTA1020250403001','FSTA1020250407009','FSTA1020250403006','FSTA1020250402008','FSTA1020250402006','FSTA1020250402007','FSTA1020250401006','FSTA1020250403003','FSTA1020250407002','FSTA1020250407005','FSTA1020250407003','FSTA1020250403005','FSTA1020250402002']"

baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
baseline = baseline_data.mean()+(baseline_data.std()*3)
baseline_block = baseline_data.mean()-(baseline_data.std()*3)

# p1.line(bin_range, baseline, line_color='green', line_width=2, line_dash='dashed')s
# p1.line(bin_range, baseline_block, line_color='red', line_width=2, line_dash='dashed')

# Plot the baselined data
stackers = df.query("Test!='TOF BASELINE'")['Stacker_SN'].unique().tolist()
for stacker in stackers:
    ys = df.query("Test!='TOF BASELINE' & Labware_Name=='nest-96-pcr' & Axis=='z' & Platform_Position=='extend' & Labware_Num_Z==1 & Zone==1 & Stacker_SN==@stacker")[bin_range]
    ys = (ys - baseline).clip(lower=0)
    ys = ys.values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color='orange',
                    line_dash='solid',
                    legend_label=stacker+'lab')

p1.legend.click_policy="hide"
show(p1)

Platform Home Baseline Samples: 90


In [39]:
p1 = figure(width=1200,height=800, title="Z Axis Baseline with 6 STD envelope, Platform extended with no labware")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]

# Plot the raw data
color_count = 0
stackers = df.query("Test=='TOF RETEST 5-30'")['Stacker_SN'].unique().tolist()
for stacker in stackers:
    ys = df.query("Test=='TOF RETEST 5-30' & Labware_Name=='baseline' & Axis=='z' & Platform_Position=='extend' & Labware_Num_X==0 & Zone==1 & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    color = Plasma256[color_count]
    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color=color,
                    line_dash='solid',
                    legend_label=stacker)
    color_count += 10
    
# Create the baseline
df_query = "Test=='TOF RETEST 5-30' & Labware_Name=='baseline' & Axis=='z' & Platform_Position=='extend' & Labware_Num_X==0 & Zone==1"
# df_query += " & Stacker_SN in ['FSTA1020250402003']"

baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
base_max = baseline_data.max()
baseline = baseline_data.mean()+(baseline_data.std()*5)
base_min = baseline_data.min()

print(np.percentile(baseline_data, 25, axis=1))
print(len(np.percentile(baseline_data, 25, axis=1)))

p1.line(bin_range, base_max, line_color='cyan', line_width=2, line_dash='dashed')
# p1.line(bin_range, baseline, line_color='green', line_width=2, line_dash='dashed')
p1.line(bin_range, base_min, line_color='red', line_width=2, line_dash='dashed')

p1.legend.click_policy="hide"
show(p1)

Platform Home Baseline Samples: 60
[52.75 50.75 49.   48.   52.   19.   16.   17.   18.75 19.   29.   31.
 31.75 30.   33.   28.   27.75 29.   27.   27.   47.75 49.75 49.   50.75
 48.   25.   26.   25.75 25.75 27.75 24.75 25.   24.75 25.75 22.75 29.
 31.   30.75 29.75 31.   23.   21.   20.75 22.75 23.   29.75 27.75 31.75
 31.   32.25 18.   20.   17.   16.75 17.75 19.   19.   19.   19.   18.75]
60


Testing if reference SPAD can correct for bin distance errors

In [24]:
p1 = figure(width=1200,height=800, title="Z Axis Baseline with 6 STD envelope, Platform extended with no labware")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]

# Plot the raw data
color_count = 0
stackers = df.query("Test=='TOF RETEST 5-30'")['Stacker_SN'].unique().tolist()
for stacker in stackers:
    ys = df.query("Test=='TOF RETEST 5-30' & Labware_Name=='baseline' & Axis=='z' & Platform_Position=='extend' & Labware_Num_X==0 & (Zone==0 | Zone==1) & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    color = Plasma256[color_count]
    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color=color,
                    line_dash='solid',
                    legend_label=stacker)
    color_count += 10
    
# Create the baseline
df_query = "Test=='TOF RETEST 5-30' & Labware_Name=='baseline' & Axis=='z' & Platform_Position=='extend' & Labware_Num_X==0 & Zone==0"
df_query += " & Stacker_SN in ['FSTA1020250407006'] & Sample==1"

baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
base_max = baseline_data.max()
baseline = baseline_data.mean()+(baseline_data.std()*5)
base_min = baseline_data.min()

# hist, bin_edge = np.histogram(baseline_data, bins=10)
# print(hist)
# print(bin_edge)

print(np.sum(baseline_data,axis=1))
new_bin_range = np.reshape(np.array(bin_range), np.shape(baseline_data))
new_weights = np.reshape(np.array(baseline_data, np.float64), np.shape(bin_range))
print(np.shape(new_bin_range))
print(np.shape(baseline_data))  

print(np.shape(bin_range))
print(np.shape(new_weights))  
print(np.array(bin_range, np.float64))
print(sum(new_weights))
bin_avg = np.average(np.array(bin_range, np.float64)[0:20], weights=new_weights[0:20])
print(bin_avg)

p1.line(bin_range, base_max, line_color='cyan', line_width=2, line_dash='dashed')
# p1.line(bin_range, baseline, line_color='green', line_width=2, line_dash='dashed')
p1.line(bin_range, base_min, line_color='red', line_width=2, line_dash='dashed')

p1.legend.click_policy="hide"
show(p1)

Platform Home Baseline Samples: 1
8971    213240.0
dtype: float64
(1, 128)
(1, 128)
(128,)
(128,)
[  1.   2.   3.   4.   5.   6.   7.   8.   9.  10.  11.  12.  13.  14.
  15.  16.  17.  18.  19.  20.  21.  22.  23.  24.  25.  26.  27.  28.
  29.  30.  31.  32.  33.  34.  35.  36.  37.  38.  39.  40.  41.  42.
  43.  44.  45.  46.  47.  48.  49.  50.  51.  52.  53.  54.  55.  56.
  57.  58.  59.  60.  61.  62.  63.  64.  65.  66.  67.  68.  69.  70.
  71.  72.  73.  74.  75.  76.  77.  78.  79.  80.  81.  82.  83.  84.
  85.  86.  87.  88.  89.  90.  91.  92.  93.  94.  95.  96.  97.  98.
  99. 100. 101. 102. 103. 104. 105. 106. 107. 108. 109. 110. 111. 112.
 113. 114. 115. 116. 117. 118. 119. 120. 121. 122. 123. 124. 125. 126.
 127. 128.]
213240.0
16.903630982996773


Testing integration algorithm

In [32]:
p1 = figure(width=1200,height=800, title="Z Axis Labware with 5 STD envelope, Platform extended, 1 Nest 96 Well PCR Plate in Z")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(15, 100)]

# Create the cumulative dataframe for the Z axis
df_query = "Test=='TOF RETEST 5-30' & Axis=='z' & Platform_Position=='extend' & Zone==1"
q_df = df.query(df_query)
cumlative_dataframe = q_df.copy()
dif_dataframe = q_df.copy()
cumlative_dataframe[bin_range] = q_df[bin_range].cumsum(axis=1)
dif_dataframe[bin_range] = cumlative_dataframe[bin_range].diff(axis=1)
# print(dif_df)

# # Plot the raw data
color_count = 0
stackers = cumlative_dataframe['Stacker_SN'].unique().tolist()
for stacker in stackers:
    ys = cumlative_dataframe.query("Labware_Name=='nest-96-pcr' & Labware_Num_Z==1 & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color='orange',
                    line_dash='solid',
                    legend_label=stacker+'lab')

    ys = cumlative_dataframe.query("Labware_Name=='baseline' & Labware_Num_Z==0 & Zone==1 & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color='blue',
                    line_dash='solid',
                    legend_label=stacker+'base')
    
    ys = dif_dataframe.query("Labware_Name=='nest-96-pcr' & Labware_Num_Z==1 & Zone==1 & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color='red',
                    line_dash='solid',
                    legend_label=stacker+'dif')
    
    ys = dif_dataframe.query("Labware_Name=='baseline' & Labware_Num_Z==0 & Zone==1 & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color='cyan',
                    line_dash='solid',
                    legend_label=stacker+'dif')
    
# # Create the baseline
df_query = "Labware_Name=='baseline' & Labware_Num_Z==0 & Zone==1"
# df_query += " & Stacker_SN in ['FSTA1020250407006'] & Sample==1"

baseline_data = cumlative_dataframe.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
mean_value = baseline_data.mean()
baseline = baseline_data.mean()+(baseline_data.std()*3)
baseline_block = baseline_data.mean()-(baseline_data.std()*3)


p1.line(bin_range, mean_value, line_color='cyan', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline, line_color='green', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_block, line_color='red', line_width=2, line_dash='dashed')

p1.legend.click_policy="hide"
show(p1)

Platform Home Baseline Samples: 60


Testing Find Peaks Algorithm

In [26]:
p1 = figure(width=1200,height=800, title="Z Axis Labware with 5 STD envelope, Platform extended, 1 Nest 96 Well PCR Plate in Z")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]

# Create the cumulative dataframe for the Z axis
df_query = "Test=='TOF RETEST 5-30' & Axis=='z' & Platform_Position=='extend' & Zone==1"
q_df = df.query(df_query)

q_df['peak_indices_detailed'] = np.ndarray
q_df['peak_heights'] = np.ndarray
q_df['peak_widths'] = np.ndarray
q_df['peak_prominence'] = np.ndarray

for index, row in q_df.iterrows():
    peaks, properties = sig.find_peaks(row[bin_range], height=200, prominence=200, width=2, rel_height=0.95)
    peaks = [x+1 for x in peaks]
    q_df.at[index, 'peak_indices_detailed'] = peaks
    q_df.at[index, 'peak_heights'] = properties['peak_heights']
    q_df.at[index, 'peak_widths'] = properties['widths']
    q_df.at[index, 'peak_prominence'] = properties['prominences']

    if row['Labware_Num_Z'] == 0:
        color= 'blue'
    else:
        color = 'orange'

    p1.scatter(x=peaks, y=properties['peak_heights'], color=color, size=properties['widths']*2)
    # p1.scatter(x=peaks, y=properties['prominences']-properties['peak_heights'], color=color, marker='x')
    p1.scatter(x=peaks, y=properties['widths']*-100, color=color, marker='x')


print(q_df[['peak_indices_detailed', 'peak_heights', 'peak_widths', 'peak_prominence']].head())

ys = q_df.query("Labware_Name=='nest-96-pcr' & Labware_Num_Z==1")[bin_range].values.tolist()
numlines = len(ys)
xs = [bin_range]*numlines

p1.multi_line(xs = xs,
                ys = ys,
                line_color='orange',
                line_dash='solid',
                legend_label='lab')

ys = q_df.query("Labware_Name=='baseline' & Labware_Num_Z==0")[bin_range].values.tolist()
numlines = len(ys)
xs = [bin_range]*numlines

p1.multi_line(xs = xs,
                ys = ys,
                line_color='blue',
                line_dash='solid',
                legend_label='base')



p1.legend.click_policy="hide"
show(p1)

/var/folders/vv/0qt52bys61x113j7x25ydnhc0000gq/T/ipykernel_36983/944248057.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  q_df['peak_indices_detailed'] = np.ndarray
/var/folders/vv/0qt52bys61x113j7x25ydnhc0000gq/T/ipykernel_36983/944248057.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  q_df['peak_heights'] = np.ndarray
/var/folders/vv/0qt52bys61x113j7x25ydnhc0000gq/T/ipykernel_36983/944248057.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try u

     peak_indices_detailed       peak_heights  \
8622              [57, 68]  [15425.0, 6128.0]   
8632              [57, 68]  [15517.0, 6169.0]   
8642              [57, 68]  [15910.0, 6322.0]   
8652              [57, 68]  [15663.0, 6240.0]   
8662              [57, 68]  [15482.0, 6231.0]   

                                  peak_widths    peak_prominence  
8622  [18.854592894417046, 6.639195225087249]  [15392.0, 5237.0]  
8632  [18.863093710799404, 6.449615327901213]  [15480.0, 5191.0]  
8642   [18.90407894797533, 6.592638662261237]  [15873.0, 5401.0]  
8652   [18.943728283777624, 6.48853789668577]  [15627.0, 5293.0]  
8662  [18.939700018942546, 6.598476221624281]  [15449.0, 5265.0]  


In [27]:
p1 = figure(width=1200,height=800, title="Z Axis Labware with 5 STD envelope, Platform extended, 1 Nest 96 Well PCR Plate in Z")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]

for index, row in q_df.iterrows():
    if row['Labware_Num_Z'] == 0:
        color= 'blue'
    else:
        color = 'orange'
    p1.scatter(x=q_df['peak_indices_detailed'], y=q_df['peak_widths'], color=color)

show(p1)